In [1]:
import pandas as pd
import numpy as np

In [31]:
movies = pd.read_csv('datos\\movies_metadata.csv', dtype={'popularity':str})
largo_original = len(movies)
pd.set_option('display.max_columns', None)
movies.head(2)


,adult,belongs_to_collection,budget,genres,homepage,id,imdb_id,original_language,original_title,overview,popularity,poster_path,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,video,vote_average,vote_count
0,False,"{'id': 10194, 'name': 'Toy Story Collection', ...",30000000,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...",http://toystory.disney.com/toy-story,862,tt0114709,en,Toy Story,"Led by Woody, Andy's toys live happily in his ...",21.946943,/rhIRbceoE9lR4veEXuwCC2wARtG.jpg,"[{'name': 'Pixar Animation Studios', 'id': 3}]","[{'iso_3166_1': 'US', 'name': 'United States o...",1995-10-30,373554033.0,81.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,NaN,Toy Story,False,7.7,5415.0
1,False,NaN,65000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",NaN,8844,tt0113497,en,Jumanji,When siblings Judy and Peter discover an encha...,17.015539,/vzmL6fP7aPKNKPRTFnZmiUfciyV.jpg,"[{'name': 'TriStar Pictures', 'id': 559}, {'na...","[{'iso_3166_1': 'US', 'name': 'United States o...",1995-12-15,262797249.0,104.0,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso...",Released,Roll the dice and unleash the excitement!,Jumanji,False,6.9,2413.0


In [11]:
movies.shape

(45466, 24)

In [ ]:
dict_booleans = ["adult", "video"]
dict_dicts = ['belongs_to_collection']
dict_numeric = ['budget', 'popularity', 'vote_average', 'vote_count']
dict_integers = ['revenue', 'runtime', ]
dict_str = ['id', 'imdb_id', 'overview', 'tagline', 'homepage', 'original_language', 'original_title', #'poster_path'
            'status', 'title']
dict_list = ['genres', 'production_companies', 'production_countries', 'spoken_languages']
dict_datetime = ['release_date']

Revisamos rápidamente los valores de 'adult' para ver si son todos booleanos, y vemos que hay otros valores, por lo que procedemos a clasificarlo como 'True' or 'False'

In [7]:
movies['adult'].unique()

array(['False', 'True', ' - Written by Ørnås',
       ' Rune Balot goes to a casino connected to the October corporation to try to wrap up her case once and for all.',
       ' Avalanche Sharks tells the story of a bikini contest that turns into a horrifying affair when it is hit by a shark avalanche.'],
      dtype=object)

Igualmente con la columna 'video', pero en esta ocasión notamos que hay un valores faltantes

In [8]:
movies['video'].unique()


array([False, True, nan], dtype=object)

In [32]:
for col in dict_str:
    movies[col] = movies[col].str.strip().str.lower()

bool_map = {'true':True, 'false':False, '- Written by Ørnås'.lower().strip():True,
            'Rune Balot goes to a casino connected to the October corporation to try to wrap up her case once and for all.'.lower().strip():False,
            'Avalanche Sharks tells the story of a bikini contest that turns into a horrifying affair when it is hit by a shark avalanche.'.lower().strip():True}

for col in dict_booleans:
    movies[col] = movies[col].astype('string').str.strip().str.lower().map(bool_map).astype('boolean')

for col in dict_numeric:
    movies[col] = pd.to_numeric(movies[col], errors='coerce')

for col in dict_integers:
    movies[col] = pd.to_numeric(movies[col], errors='coerce')

for col in dict_datetime:
    movies[col] = pd.to_datetime(movies[col], errors='coerce')

In [13]:
def tipo_dato_x_columna(tabla):
    tipos = tabla.dtypes.reset_index()
    tipos.columns = ['columna', 'tipo de dato']
    return tipos

def nulos_x_columna(tabla):
    cantidad_nulos = tabla.isnull().sum().reset_index()
    cantidad_nulos.columns = ['columna', 'cantidad de valores nulos']
    return cantidad_nulos

def unicos_x_columna(tabla):
    cantidad_unicos = tabla.nunique().reset_index()
    cantidad_unicos.columns = ['columna', 'cantidad de valores unicos']
    return cantidad_unicos

def tipo_nulo_unicos_x_columna(tabla):
    tipo = tipo_dato_x_columna(tabla)
    nulos = nulos_x_columna(tabla)
    unicos = unicos_x_columna(tabla)
    datos = [df.set_index('columna') for df in [tipo, nulos, unicos]]

    union = pd.concat(datos, axis=1, join='inner').reset_index()
    return union

Verificamos el tipo de variable, que tantos nulos hay por columna, asi como sus valores únicos.

In [15]:
tipo_nulo_unicos_x_columna(movies)

,columna,tipo de dato,cantidad de valores nulos,cantidad de valores unicos
0,adult,boolean,0,2
1,belongs_to_collection,object,40972,1698
2,budget,float64,3,1223
3,genres,object,0,4069
4,homepage,object,37684,7673
5,id,object,0,45436
6,imdb_id,object,17,45417
7,original_language,object,11,92
8,original_title,object,0,43330
9,overview,object,954,44306


Ahora vamos a ver que duplicados exactos hay sin contar la columna de 'video' para saber si en esos registros, tenemos que hacer algún tipo de imputación en los valores faltantes de 'video'.
Pero notamos que no hace falta realizar ningún tipo de imputación ya que los registros duplicados no tiene ningun valor faltante en video, por lo que la ausencia de 'video' no es un factor que afecte a duplicados

In [30]:
sin_video = movies.columns.difference(['video'])
movies[movies.duplicated(subset=sin_video, keep=False)][['video', 'title']].sort_values(by='title')

,video,title
949,False,a farewell to arms
15074,False,a farewell to arms
21116,False,a place at the table
2564,False,a place at the table
11155,False,black gold
20843,False,black gold
13375,False,blackout
13261,False,blackout
16764,False,blackout
23044,False,brotherhood


De la misma manera, buscaremos duplicados de sin tomar en cuenta la columna de 'imdb_id' ya que igualmente esta columna tiene valores faltantes y dependiendo de esto sabremos si debemos de buscar su 'imdb_id'.
Pero notamos de la misma manera que la ausencia de 'imdb_id' no influye si 2 registros son exactamente igual, por lo que no se hará la imputación.

In [25]:
sin_imdb_id = movies.columns.difference(['imdb_id'])
movies[movies.duplicated(subset=sin_imdb_id, keep=False)][['imdb_id', 'title']].sort_values(by='title')

,imdb_id,title
13261,tt1180333,blackout
13375,tt1180333,blackout
16764,tt1180333,blackout
17229,tt1327820,brotherhood
23044,tt1327820,brotherhood
40040,tt2818654,cemetery of splendour
33184,tt2818654,cemetery of splendour
22151,tt0499456,days of darkness
14000,tt0499456,days of darkness
24844,tt0446676,deal


Como sabemos que hay diferentes maneras de la calificación de 'popularity', vamos a ver si 'popularity' es un factor que influye en la exactitud de los registros.
Notamos que efectivamente hay 2 registros exactamente duplicados pero diferente 'popularity', y como no hay un definición clara para medirla, lo que se tomará aquí es que se agrupara por 'id' de pelicula, una vez agrupados se tomará el promedio, si todos los registros duplicados tienen el mismo 'popularity' no le afectará, si tiene diferente se cambiará al promedio, de está manera se corrigen los problema de incosistencia en 'popularity'

In [33]:
sin_popularity = movies.columns.difference(['popularity'])
movies[movies.duplicated(subset=sin_popularity, keep=False)][['popularity', 'title']].sort_values(by='title')

,popularity,title
949,1.914697,a farewell to arms
15074,2.411191,a farewell to arms
21116,1.673307,a place at the table
2564,0.501046,a place at the table
11155,6.652197,black gold
20843,6.475665,black gold
13375,0.411949,blackout
13261,0.411949,blackout
16764,0.411949,blackout
23044,2.587911,brotherhood


In [ ]:
movies['popularity'] = movies.groupby('id')['popularity'].transform('mean')
movies[movies.duplicated(subset=sin_popularity, keep=False)][['popularity', 'title']].sort_values(by='title')

De la misma manera observamos la columna de fechas para saber si es necesario realizar algún tipo de imputación o corrección de ellas.
Observando los duplicados exactos sin tomar en cuenta la fecha, vemos que la fecha es la misma, por lo que la fecha no juega un papel en los duplicados.

In [34]:
sin_date = movies.columns.difference(['release_date'])
movies[movies.duplicated(subset=sin_date, keep=False)][['release_date', 'title']].sort_values(by='title')

,release_date,title
13261,2008-12-26,blackout
13375,2008-12-26,blackout
16764,2008-12-26,blackout
17229,2009-10-21,brotherhood
23044,2009-10-21,brotherhood
40040,2015-09-02,cemetery of splendour
33184,2015-09-02,cemetery of splendour
22151,2007-01-01,days of darkness
14000,2007-01-01,days of darkness
24844,2008-01-29,deal


In [35]:
sin_budget = movies.columns.difference(['budget'])
movies[movies.duplicated(subset=sin_budget, keep=False)][['budget', 'title']].sort_values(by='title')

,budget,title
13261,0.0,blackout
13375,0.0,blackout
16764,0.0,blackout
17229,0.0,brotherhood
23044,0.0,brotherhood
40040,980000.0,cemetery of splendour
33184,980000.0,cemetery of splendour
22151,0.0,days of darkness
14000,0.0,days of darkness
24844,0.0,deal


In [ ]:
sin_ = movies.columns.difference(['original_language'])
movies[movies.duplicated(subset=sin_, keep=False)][['original_language', 'title']].sort_values(by='title')

,original_language,title
13261,fi,blackout
13375,fi,blackout
16764,fi,blackout
17229,da,brotherhood
23044,da,brotherhood
40040,th,cemetery of splendour
33184,th,cemetery of splendour
22151,en,days of darkness
14000,en,days of darkness
24844,en,deal


In [37]:
sin_ = movies.columns.difference(['original_title'])
movies[movies.duplicated(subset=sin_, keep=False)][['original_title', 'title']].sort_values(by='title')

,original_title,title
13261,blackout,blackout
13375,blackout,blackout
16764,blackout,blackout
17229,broderskab,brotherhood
23044,broderskab,brotherhood
40040,รักที่ขอนแก่น,cemetery of splendour
33184,รักที่ขอนแก่น,cemetery of splendour
22151,days of darkness,days of darkness
14000,days of darkness,days of darkness
24844,deal,deal


In [38]:
sin_ = movies.columns.difference(['overview'])
movies[movies.duplicated(subset=sin_, keep=False)][['overview', 'title']].sort_values(by='title')

,overview,title
13261,recovering from a nail gun shot to the head an...,blackout
13375,recovering from a nail gun shot to the head an...,blackout
16764,recovering from a nail gun shot to the head an...,blackout
17229,former danish servicemen lars and jimmy are th...,brotherhood
23044,former danish servicemen lars and jimmy are th...,brotherhood
40040,"in a hospital, ten soldiers are being treated ...",cemetery of splendour
33184,"in a hospital, ten soldiers are being treated ...",cemetery of splendour
22151,when a comet strikes earth and kicks up a clou...,days of darkness
14000,when a comet strikes earth and kicks up a clou...,days of darkness
24844,as an ex-gambler teaches a hot-shot college ki...,deal


In [39]:
sin_ = movies.columns.difference(['tagline'])
movies[movies.duplicated(subset=sin_, keep=False)][['tagline', 'title']].sort_values(by='title')

,tagline,title
13261,which one is the first to return - memory or t...,blackout
13375,which one is the first to return - memory or t...,blackout
16764,which one is the first to return - memory or t...,blackout
17229,NaN,brotherhood
23044,NaN,brotherhood
40040,NaN,cemetery of splendour
33184,NaN,cemetery of splendour
22151,NaN,days of darkness
14000,NaN,days of darkness
24844,NaN,deal


In [40]:
sin_ = movies.columns.difference(['revenue'])
movies[movies.duplicated(subset=sin_, keep=False)][['revenue', 'title']].sort_values(by='title')

,revenue,title
13261,0.0,blackout
13375,0.0,blackout
16764,0.0,blackout
17229,0.0,brotherhood
23044,0.0,brotherhood
40040,0.0,cemetery of splendour
33184,0.0,cemetery of splendour
22151,0.0,days of darkness
14000,0.0,days of darkness
24844,0.0,deal


In [41]:
sin_ = movies.columns.difference(['vote_average'])
movies[movies.duplicated(subset=sin_, keep=False)][['vote_average', 'title']].sort_values(by='title')

,vote_average,title
13261,6.7,blackout
13375,6.7,blackout
16764,6.7,blackout
17229,7.1,brotherhood
23044,7.1,brotherhood
40040,4.4,cemetery of splendour
33184,4.4,cemetery of splendour
22151,5.0,days of darkness
14000,5.0,days of darkness
24844,5.2,deal


In [42]:
sin_ = movies.columns.difference(['vote_count'])
movies[movies.duplicated(subset=sin_, keep=False)][['vote_count', 'title']].sort_values(by='title')

,vote_count,title
13261,3.0,blackout
13375,3.0,blackout
16764,3.0,blackout
17229,21.0,brotherhood
23044,21.0,brotherhood
40040,50.0,cemetery of splendour
33184,50.0,cemetery of splendour
22151,5.0,days of darkness
14000,5.0,days of darkness
24844,22.0,deal


In [43]:
sin_ = movies.columns.difference(['runtime'])
movies[movies.duplicated(subset=sin_, keep=False)][['runtime', 'title']].sort_values(by='title')

,runtime,title
13261,108.0,blackout
13375,108.0,blackout
16764,108.0,blackout
17229,90.0,brotherhood
23044,90.0,brotherhood
40040,122.0,cemetery of splendour
33184,122.0,cemetery of splendour
22151,89.0,days of darkness
14000,89.0,days of darkness
24844,85.0,deal


por lo que ahora vamos a eliminar dupplicados de solo ciertas columnas, las que creo que son más importantes y que se resolvieron y analizaron inconssitencias, ya que las demás como llegan a ser objetos como listas o diccionarios es más complejo de limpiar, además estas son las que a mi parecer toman más peso

In [44]:
columnas = ['original_language', 'original_title', 'overview', 'tagline', 'title', 'imdb_id', 'release_date', 'budget', 'id', 'revenue', 'runtime', 'vote_average', 'vote_count', 'popularity']

movies = movies.drop_duplicates(subset=columnas, keep='first')
largo_despues = len(movies)

In [45]:
print(f'Se quitaron {largo_original-largo_despues} filas duplicadas')

Se quitaron 17 filas duplicadas
